# Channel 2 Cilia U-Net Segmentation, PyTorch and FastAPI

This notebook trains and serves a binary semantic segmentation model for cilia. It uses **only the second image channel** as model input everywhere: training, validation, testing, prediction, visualization, and API inference.

The second channel means zero-based `CHANNEL_INDEX = 1`. After each image is standardized to `(height, width, channels)`, the notebook selects channel 2 exactly like this:

```python
channel2 = image[..., 1:2]
```

No other image channel is merged, averaged, normalized, resized, displayed, or sent to the U-Net.

## Expected Folder Structure

Put paired training images and manually prepared binary cilia masks in separate folders. Image and mask files must have matching filename stems.

```text
/content/drive/MyDrive/cilia-detector-2/
  data/
    images/
      CS02-001.tif
      CS02-002.tif
    masks/
      CS02-001.tif
      CS02-002.tif
    test_images/              # optional held-out test images
      CS02-101.tif
    test_masks/               # optional held-out test masks
      CS02-101.tif
  checkpoints/
```

Supported image formats: `.tif`, `.tiff`, `.npy`, `.npz`, `.png`, `.jpg`, `.jpeg`.

Supported image shape/channel order:

- `(height, width, channels)` / HWC
- `(channels, height, width)` / CHW, common for ImageJ multipage TIFFs

Masks should be 2D binary images where `cilia = 1` or any positive value, and `background = 0`.

## Install Dependencies and Mount Drive

Run this cell first in Google Colab. If Colab already has PyTorch installed, pip will usually keep the existing GPU-enabled version.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

!pip install -q tifffile scikit-image pillow matplotlib fastapi uvicorn python-multipart pyngrok requests

## Configuration

These variables control the folders, model path, image size, and selected input channel. Keep `CHANNEL_INDEX = 1` for the second channel.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/cilia-detector-2')
IMAGE_DIR = PROJECT_DIR / 'data/images'
MASK_DIR = PROJECT_DIR / 'data/masks'
TEST_IMAGE_DIR = PROJECT_DIR / 'data/test_images'  # optional held-out test images
TEST_MASK_DIR = PROJECT_DIR / 'data/test_masks'    # optional held-out test masks
IMAGE_SIZE = (256, 256)  # (height, width)
CHANNEL_INDEX = 1        # zero-based index; 1 means second channel
MODEL_PATH = PROJECT_DIR / 'checkpoints/channel2_cilia_unet.pt'

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
print('IMAGE_DIR:', IMAGE_DIR)
print('MASK_DIR:', MASK_DIR)
print('MODEL_PATH:', MODEL_PATH)

## Imports

The U-Net is implemented in PyTorch. Images are loaded with `tifffile`, NumPy, or Pillow depending on file format.

In [ ]:
import io
import json
import shutil
import tempfile
import zipfile
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from skimage.transform import resize
import tifffile as tiff
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split

SUPPORTED_IMAGE_EXTENSIONS = {'.tif', '.tiff', '.npy', '.npz', '.png', '.jpg', '.jpeg'}
SUPPORTED_MASK_EXTENSIONS = {'.tif', '.tiff', '.npy', '.npz', '.png', '.jpg', '.jpeg'}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch device:', device)

## Loading and Preprocessing

This is the important channel-handling section. The image is first standardized to HWC. Then only channel index 1 is selected with `image[..., 1:2]`. Normalization and resizing happen after that selection, so the other channels are never processed for model input.

In [ ]:
def read_array(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {'.tif', '.tiff'}:
        return np.asarray(tiff.imread(path))
    if suffix == '.npy':
        return np.asarray(np.load(path))
    if suffix == '.npz':
        data = np.load(path)
        if len(data.files) != 1:
            raise ValueError(f'{path.name} has multiple arrays. Use a single-array .npz file.')
        return np.asarray(data[data.files[0]])
    if suffix in {'.png', '.jpg', '.jpeg'}:
        return np.asarray(Image.open(path))
    raise ValueError(f'Unsupported file format: {suffix}')


def standardize_multichannel_to_hwc(image):
    arr = np.asarray(image)
    if arr.ndim > 3:
        squeezed = np.squeeze(arr)
        if squeezed.ndim != arr.ndim:
            arr = squeezed

    if arr.ndim == 2:
        return arr[..., np.newaxis]
    if arr.ndim != 3:
        raise ValueError(
            f'Expected a 2D image or 3D multichannel image, got shape {arr.shape}. '
            'Supported channel orders are HWC and CHW.'
        )

    if arr.shape[-1] <= 16:
        return arr
    if arr.shape[0] <= 16:
        return np.moveaxis(arr, 0, -1)
    raise ValueError(
        f'Could not infer channel axis for shape {arr.shape}. Expected HWC or CHW with a small channel dimension.'
    )


def select_channel_preserve_dim(image, channel_index=CHANNEL_INDEX):
    hwc = standardize_multichannel_to_hwc(image)
    if channel_index < 0 or channel_index >= hwc.shape[-1]:
        raise ValueError(
            f'Requested CHANNEL_INDEX={channel_index}, but the image has {hwc.shape[-1]} channel(s). '
            'Channel 2 input requires at least 2 channels.'
        )
    # This is the required second-channel-only extraction.
    return hwc[..., channel_index:channel_index + 1].astype(np.float32)


def normalize_selected_channel(channel_hwc, lower=1.0, upper=99.8):
    channel_hwc = channel_hwc.astype(np.float32)
    lo, hi = np.percentile(channel_hwc, (lower, upper))
    if hi <= lo:
        return np.zeros_like(channel_hwc, dtype=np.float32)
    normalized = (channel_hwc - lo) / (hi - lo)
    return np.clip(normalized, 0.0, 1.0).astype(np.float32)


def preprocess_image_array(image, image_size=IMAGE_SIZE, channel_index=CHANNEL_INDEX):
    channel = select_channel_preserve_dim(image, channel_index=channel_index)
    channel = normalize_selected_channel(channel)
    resized = resize(
        channel,
        (int(image_size[0]), int(image_size[1]), 1),
        order=1,
        preserve_range=True,
        anti_aliasing=True,
    )
    return resized.astype(np.float32)


def preprocess_image_file(image_path, image_size=IMAGE_SIZE, channel_index=CHANNEL_INDEX):
    return preprocess_image_array(read_array(image_path), image_size=image_size, channel_index=channel_index)


def preprocess_mask_array(mask, image_size=IMAGE_SIZE):
    arr = np.asarray(mask)
    arr = np.squeeze(arr)
    if arr.ndim == 3:
        if arr.shape[-1] <= 4:
            arr = arr[..., 0]
        elif arr.shape[0] <= 4:
            arr = arr[0]
        else:
            raise ValueError(f'Mask shape {arr.shape} is ambiguous. Use a 2D binary mask.')
    if arr.ndim != 2:
        raise ValueError(f'Expected a 2D binary mask, got shape {arr.shape}.')

    binary = (arr > 0).astype(np.float32)
    resized = resize(
        binary,
        (int(image_size[0]), int(image_size[1])),
        order=0,
        preserve_range=True,
        anti_aliasing=False,
    )
    return (resized > 0.5).astype(np.float32)[..., np.newaxis]


def preprocess_mask_file(mask_path, image_size=IMAGE_SIZE):
    return preprocess_mask_array(read_array(mask_path), image_size=image_size)

## Paired Folder Loader

This single function loads training images and masks from folders. It matches files by filename stem and prints a helpful error if a training image is missing a mask.

In [ ]:
def collect_files_by_stem(folder, extensions):
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f'Folder does not exist: {folder}')
    files = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in extensions]
    by_stem = {}
    duplicates = {}
    for path in files:
        if path.stem in by_stem:
            duplicates.setdefault(path.stem, [by_stem[path.stem].name]).append(path.name)
        by_stem[path.stem] = path
    if duplicates:
        details = '\n'.join(f'- {stem}: {", ".join(names)}' for stem, names in duplicates.items())
        raise ValueError(f'Duplicate filename stems found. Keep one file per stem:\n{details}')
    return by_stem


def find_image_mask_pairs(image_dir=IMAGE_DIR, mask_dir=MASK_DIR):
    image_map = collect_files_by_stem(image_dir, SUPPORTED_IMAGE_EXTENSIONS)
    mask_map = collect_files_by_stem(mask_dir, SUPPORTED_MASK_EXTENSIONS)

    if not image_map:
        raise FileNotFoundError(f'No supported image files found in {image_dir}')
    if not mask_map:
        raise FileNotFoundError(f'No supported mask files found in {mask_dir}')

    missing_masks = [path.name for stem, path in sorted(image_map.items()) if stem not in mask_map]
    if missing_masks:
        missing_text = '\n'.join(f'- {name}' for name in missing_masks)
        raise FileNotFoundError(
            'Every training image needs a matching mask with the same filename stem.\n'
            'Example: images/CS02-001.tif matches masks/CS02-001.tif.\n\n'
            f'Images missing masks:\n{missing_text}'
        )

    return [(stem, image_path, mask_map[stem]) for stem, image_path in sorted(image_map.items())]


def load_paired_training_images_and_masks(
    image_dir=IMAGE_DIR,
    mask_dir=MASK_DIR,
    image_size=IMAGE_SIZE,
    channel_index=CHANNEL_INDEX,
):
    pairs = find_image_mask_pairs(image_dir, mask_dir)
    images = []
    masks = []
    for stem, image_path, mask_path in pairs:
        images.append(preprocess_image_file(image_path, image_size=image_size, channel_index=channel_index))
        masks.append(preprocess_mask_file(mask_path, image_size=image_size))
    X = np.stack(images).astype(np.float32)
    y = np.stack(masks).astype(np.float32)
    print(f'Loaded {len(pairs)} image/mask pairs')
    print('X shape:', X.shape, 'meaning (N, height, width, 1)')
    print('y shape:', y.shape, 'meaning binary cilia masks')
    return X, y, pairs

## One-Channel PyTorch U-Net

The preprocessed data shape is `(height, width, 1)`. PyTorch convolution layers expect `(batch, channels, height, width)`, so the dataset transposes each image to `(1, height, width)` before passing it to the model.

This is binary semantic segmentation: cilia pixels are `1`, background pixels are `0`. The model outputs one sigmoid probability map.

In [ ]:
class CiliaDataset(Dataset):
    def __init__(self, images, masks=None, augment=False):
        self.images = images
        self.masks = masks
        self.augment = augment
        if masks is not None and len(images) != len(masks):
            raise ValueError('images and masks must contain the same number of items')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = None if self.masks is None else self.masks[idx]
        if self.augment and mask is not None:
            if np.random.random() < 0.5:
                image = np.flip(image, axis=1).copy()
                mask = np.flip(mask, axis=1).copy()
            if np.random.random() < 0.5:
                image = np.flip(image, axis=0).copy()
                mask = np.flip(mask, axis=0).copy()

        image_tensor = torch.from_numpy(np.moveaxis(image, -1, 0)).float()
        if mask is None:
            return image_tensor
        mask_tensor = torch.from_numpy(np.moveaxis(mask, -1, 0)).float()
        return image_tensor, mask_tensor


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=(32, 64, 128, 256)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        channels = in_channels
        for feature in features:
            self.downs.append(ConvBlock(channels, feature))
            channels = feature
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(features[-1], features[-1] * 2)
        channels = features[-1] * 2
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(channels, feature, kernel_size=2, stride=2))
            self.ups.append(ConvBlock(feature * 2, feature))
            channels = feature
        self.head = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip = skips[idx // 2]
            if x.shape != skip.shape:
                x = nn.functional.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x = torch.cat([skip, x], dim=1)
            x = self.ups[idx + 1](x)
        return self.head(x)


def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    intersection = torch.sum(probs * targets)
    denominator = torch.sum(probs) + torch.sum(targets)
    return 1 - (2 * intersection + eps) / (denominator + eps)


def dice_iou_from_logits(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    targets = targets.float()
    intersection = torch.sum(preds * targets)
    pred_sum = torch.sum(preds)
    target_sum = torch.sum(targets)
    union = pred_sum + target_sum - intersection
    dice = (2 * intersection + eps) / (pred_sum + target_sum + eps)
    iou = (intersection + eps) / (union + eps)
    return float(dice.item()), float(iou.item())

## Training and Validation

The loss combines binary cross-entropy and Dice loss. This is useful because cilia are thin and occupy a small fraction of each image. The helper below can train a new one-channel U-Net or fine-tune an existing one-channel U-Net passed through `model=`.


In [ ]:
def train_model(
    X,
    y,
    epochs,
    batch_size,
    learning_rate=1e-3,
    val_fraction=0.2,
    model_path=MODEL_PATH,
    model=None,
    overwrite_model=False,
):
    if epochs <= 0:
        raise ValueError('epochs must be a positive integer.')
    if batch_size <= 0:
        raise ValueError('batch_size must be a positive integer.')
    if learning_rate <= 0:
        raise ValueError('learning_rate must be positive.')
    if not (0 <= val_fraction < 1):
        raise ValueError('val_fraction must be at least 0 and less than 1.')
    if X.ndim != 4 or X.shape[-1] != 1:
        raise ValueError(f'The U-Net expects one-channel images with shape (N, H, W, 1), got X shape {X.shape}')
    if y.ndim != 4 or y.shape[-1] != 1:
        raise ValueError(f'Masks should have shape (N, H, W, 1), got y shape {y.shape}')

    if model_path is not None:
        model_path = Path(model_path)
        if model_path.exists() and not overwrite_model:
            raise FileExistsError(
                f'Model already exists at {model_path}. To avoid accidental overwrites, choose a new MODEL_PATH '
                'or pass overwrite_model=True after you are sure you want to replace it.'
            )
        model_path.parent.mkdir(parents=True, exist_ok=True)

    dataset = CiliaDataset(X, y, augment=True)
    if len(dataset) > 1 and val_fraction > 0:
        val_size = max(1, int(len(dataset) * val_fraction))
        train_size = len(dataset) - val_size
        if train_size <= 0:
            train_size = len(dataset) - 1
            val_size = 1
        train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
    else:
        train_ds = dataset
        val_ds = dataset

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = model if model is not None else UNet(in_channels=1, out_channels=1)
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([6.0], device=device))

    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = bce(logits, masks) + dice_loss(logits, masks)
            loss.backward()
            optimizer.step()
            train_loss += float(loss.item()) * images.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        dice_total = 0.0
        iou_total = 0.0
        seen = 0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                logits = model(images)
                loss = bce(logits, masks) + dice_loss(logits, masks)
                batch_n = images.size(0)
                val_loss += float(loss.item()) * batch_n
                dice, iou = dice_iou_from_logits(logits, masks)
                dice_total += dice * batch_n
                iou_total += iou * batch_n
                seen += batch_n
        val_loss /= len(val_loader.dataset)
        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_dice': dice_total / max(seen, 1),
            'val_iou': iou_total / max(seen, 1),
        }
        history.append(row)
        print(
            f"epoch={epoch:03d} train_loss={row['train_loss']:.4f} "
            f"val_loss={row['val_loss']:.4f} val_dice={row['val_dice']:.4f} val_iou={row['val_iou']:.4f}"
        )

    if model_path is not None:
        torch.save({
            'model_state': model.state_dict(),
            'in_channels': 1,
            'image_size': tuple(X.shape[1:3]),
            'channel_index': CHANNEL_INDEX,
            'threshold': 0.5,
        }, model_path)
        print('Saved model to', model_path)
    return model, history


def load_model(model_path=MODEL_PATH):
    checkpoint = torch.load(model_path, map_location=device)
    if int(checkpoint.get('in_channels', 1)) != 1:
        raise ValueError('This notebook requires a one-channel model. Retrain with CHANNEL_INDEX=1.')
    if int(checkpoint.get('channel_index', CHANNEL_INDEX)) != CHANNEL_INDEX:
        raise ValueError(
            f'This notebook is configured for CHANNEL_INDEX={CHANNEL_INDEX}, but the checkpoint was saved with '
            f"channel_index={checkpoint.get('channel_index')}."
        )
    model = UNet(in_channels=1, out_channels=1).to(device)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    return model, checkpoint


def predict_preprocessed(model, channel2_hwc, threshold=0.5):
    if channel2_hwc.shape[-1] != 1:
        raise ValueError(f'Expected selected channel shape (H, W, 1), got {channel2_hwc.shape}')
    tensor = torch.from_numpy(np.moveaxis(channel2_hwc, -1, 0)[np.newaxis]).float().to(device)
    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(tensor))[0, 0].cpu().numpy()
    pred_mask = (prob >= threshold).astype(np.uint8)
    return prob, pred_mask


## Train the Model

Training requires multichannel images and matching manually prepared cilia masks. This cell will not start training until `CONFIRM_TRAIN = True`, and it will not overwrite an existing model unless `OVERWRITE_MODEL = True`. If you only want prediction with an existing model, leave `CONFIRM_TRAIN = False` and skip ahead.


In [ ]:
# Adjust these explicit parameters before running training.
CONFIRM_TRAIN = False
EPOCHS = 30
BATCH_SIZE = 4
LEARNING_RATE = 1e-3
OVERWRITE_MODEL = False

if not CONFIRM_TRAIN:
    print('Training skipped. Set CONFIRM_TRAIN = True after IMAGE_DIR and MASK_DIR contain matching images and masks.')
else:
    X, y, pairs = load_paired_training_images_and_masks(
        image_dir=IMAGE_DIR,
        mask_dir=MASK_DIR,
        image_size=IMAGE_SIZE,
        channel_index=CHANNEL_INDEX,
    )

    model, history = train_model(
        X,
        y,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        model_path=MODEL_PATH,
        overwrite_model=OVERWRITE_MODEL,
    )


## Visualization

Every visualization below shows only the allowed panels:

- `Channel 2 Input`
- `True Cilia Mask`, when available
- `Predicted Cilia Mask`

The optional overlay uses the selected channel 2 image only.

In [ ]:
def show_prediction(channel2_hwc, true_mask=None, pred_mask=None, show_overlay=True):
    panels = [('Channel 2 Input', np.squeeze(channel2_hwc), 'gray')]
    if true_mask is not None:
        panels.append(('True Cilia Mask', np.squeeze(true_mask), 'gray'))
    if pred_mask is not None:
        panels.append(('Predicted Cilia Mask', np.squeeze(pred_mask), 'gray'))

    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
    if len(panels) == 1:
        axes = [axes]
    for ax, (title, image, cmap) in zip(axes, panels):
        ax.imshow(image, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    plt.show()

    if show_overlay and pred_mask is not None:
        plt.figure(figsize=(5, 5))
        plt.imshow(np.squeeze(channel2_hwc), cmap='gray')
        plt.imshow(np.squeeze(pred_mask), cmap='Reds', alpha=0.35)
        plt.title('Predicted Cilia Mask Overlay on Channel 2 Input')
        plt.axis('off')
        plt.show()


# Example validation display using the first loaded pair.
if 'model' in globals() and len(X) > 0:
    probability, pred = predict_preprocessed(model, X[0], threshold=0.5)
    show_prediction(X[0], true_mask=y[0], pred_mask=pred, show_overlay=True)

## Optional Test Set Evaluation

Use this section when you have a held-out test set with images and manually prepared masks. It uses the same folder-pair loader as training, so each test image is standardized, reduced to `image[..., 1:2]`, normalized, and resized before evaluation. The reported test metrics are Dice coefficient and IoU.


In [ ]:
def evaluate_model_on_arrays(model, X_eval, y_eval, batch_size=4, threshold=0.5):
    if X_eval.ndim != 4 or X_eval.shape[-1] != 1:
        raise ValueError(f'Expected one-channel test images with shape (N, H, W, 1), got {X_eval.shape}')
    if y_eval.ndim != 4 or y_eval.shape[-1] != 1:
        raise ValueError(f'Expected binary test masks with shape (N, H, W, 1), got {y_eval.shape}')

    loader = DataLoader(CiliaDataset(X_eval, y_eval, augment=False), batch_size=batch_size, shuffle=False)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([6.0], device=device))
    model.eval()
    total_loss = 0.0
    dice_total = 0.0
    iou_total = 0.0
    seen = 0
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            logits = model(images)
            loss = bce(logits, masks) + dice_loss(logits, masks)
            batch_n = images.size(0)
            total_loss += float(loss.item()) * batch_n
            dice, iou = dice_iou_from_logits(logits, masks, threshold=threshold)
            dice_total += dice * batch_n
            iou_total += iou * batch_n
            seen += batch_n
    return {
        'test_loss': total_loss / max(seen, 1),
        'test_dice': dice_total / max(seen, 1),
        'test_iou': iou_total / max(seen, 1),
        'test_images': seen,
    }


RUN_TEST_EVALUATION = False
TEST_BATCH_SIZE = 4
TEST_THRESHOLD = 0.5

if not RUN_TEST_EVALUATION:
    print('Test evaluation skipped. Set RUN_TEST_EVALUATION = True after TEST_IMAGE_DIR and TEST_MASK_DIR contain matching files.')
else:
    model, checkpoint = load_model(MODEL_PATH)
    X_test, y_test, test_pairs = load_paired_training_images_and_masks(
        image_dir=TEST_IMAGE_DIR,
        mask_dir=TEST_MASK_DIR,
        image_size=tuple(checkpoint.get('image_size', IMAGE_SIZE)),
        channel_index=CHANNEL_INDEX,
    )
    test_metrics = evaluate_model_on_arrays(
        model,
        X_test,
        y_test,
        batch_size=TEST_BATCH_SIZE,
        threshold=TEST_THRESHOLD,
    )
    print(json.dumps(test_metrics, indent=2))
    probability, pred = predict_preprocessed(model, X_test[0], threshold=TEST_THRESHOLD)
    show_prediction(X_test[0], true_mask=y_test[0], pred_mask=pred, show_overlay=True)


## Predict a New Image

Prediction requires only a new multichannel image. The notebook extracts channel index 1, preprocesses it exactly like training, and predicts a binary cilia mask.

In [ ]:
# Example: change this path to a new multichannel bio-image.
NEW_IMAGE_PATH = IMAGE_DIR / 'example_new_image.tif'

if NEW_IMAGE_PATH.exists():
    model, checkpoint = load_model(MODEL_PATH)
    channel2 = preprocess_image_file(NEW_IMAGE_PATH, image_size=IMAGE_SIZE, channel_index=CHANNEL_INDEX)
    probability, predicted_mask = predict_preprocessed(model, channel2, threshold=checkpoint.get('threshold', 0.5))
    show_prediction(channel2, true_mask=None, pred_mask=predicted_mask, show_overlay=True)
else:
    print('Set NEW_IMAGE_PATH to a real image before running prediction.')

## FastAPI Service for Colab

This API can be exposed with ngrok. It uses the same preprocessing functions above, so `/predict` and `/train` also use only zero-based channel index 1.

Supported upload formats are `.tif`, `.tiff`, `.npy`, `.npz`, `.png`, `.jpg`, and `.jpeg`. Images must be multichannel arrays in `(height, width, channels)` or `(channels, height, width)` order with at least two channels. Masks must be 2D binary images where cilia are positive pixels and background is zero.

Endpoints:

- `GET /health`: confirms the API is running.
- `POST /predict`: accepts one multichannel image and returns a PNG mask, or a ZIP with mask plus a channel-2-only overlay.
- `POST /train`: accepts a ZIP with `images/` and `masks/`, validates pairs, fine-tunes the existing one-channel U-Net, saves the updated model, and returns metrics.

Training requires images and matching manually prepared cilia masks. Detection/prediction requires only new images.


In [ ]:
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import JSONResponse, StreamingResponse

app = FastAPI(
    title='Channel 2 Cilia Segmentation API',
    description=(
        'Binary cilia segmentation for multichannel bio-images. The API always extracts zero-based '
        'channel index 1, preserves it as one channel, and never uses other channels for the model input.'
    ),
)


def mask_png_bytes(mask):
    buffer = io.BytesIO()
    Image.fromarray((mask.astype(bool).astype(np.uint8) * 255), mode='L').save(buffer, format='PNG')
    return buffer.getvalue()


def overlay_png_bytes(channel2_hwc, mask, alpha=0.45):
    if channel2_hwc.ndim != 3 or channel2_hwc.shape[-1] != 1:
        raise ValueError(f'Overlay expects selected Channel 2 Input with shape (H, W, 1), got {channel2_hwc.shape}.')
    base = (np.squeeze(channel2_hwc).clip(0, 1) * 255).astype(np.uint8)
    rgb = np.stack([base, base, base], axis=-1).astype(np.float32)
    red = np.zeros_like(rgb)
    red[..., 0] = 255
    mask_bool = mask.astype(bool)
    rgb[mask_bool] = (1 - alpha) * rgb[mask_bool] + alpha * red[mask_bool]
    buffer = io.BytesIO()
    Image.fromarray(np.clip(rgb, 0, 255).astype(np.uint8), mode='RGB').save(buffer, format='PNG')
    return buffer.getvalue()


def safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise HTTPException(status_code=400, detail='ZIP file contains an unsafe path. Please remove absolute paths or ../ entries.')
        zf.extractall(destination)


def find_training_dirs(extract_dir):
    extract_dir = Path(extract_dir)
    candidates = [extract_dir] + [p for p in extract_dir.iterdir() if p.is_dir()]
    for base in candidates:
        image_dir = base / 'images'
        mask_dir = base / 'masks'
        if image_dir.is_dir() and mask_dir.is_dir():
            return image_dir, mask_dir
    raise HTTPException(
        status_code=400,
        detail=(
            'Could not find paired folders named images/ and masks/ inside the ZIP file. Example ZIP layout: '
            'training_pairs/images/sample_001.tif and training_pairs/masks/sample_001.tif.'
        ),
    )


@app.get('/health')
def health():
    return {
        'status': 'ok',
        'model_path': str(MODEL_PATH),
        'model_exists': MODEL_PATH.exists(),
        'channel_index': CHANNEL_INDEX,
        'channel_name': 'Channel 2 Input',
        'image_size': IMAGE_SIZE,
        'supported_formats': sorted(SUPPORTED_IMAGE_EXTENSIONS),
        'expected_image_order': 'HWC or CHW multichannel image with at least two channels',
    }


@app.post('/predict')
async def api_predict(
    file: UploadFile = File(...),
    threshold: float = Form(0.5),
    return_overlay: bool = Form(True),
):
    """Predict cilia from one multichannel image.

    Upload a TIFF/TIF, NPY, NPZ, PNG, JPG, or JPEG image in HWC or CHW channel order.
    The API extracts only zero-based channel index 1 as image[..., 1:2], normalizes and resizes that selected channel,
    and returns predicted_cilia_mask.png. If return_overlay=true, the response is a ZIP that also includes a transparent
    predicted-mask overlay on the Channel 2 Input.
    """
    suffix = Path(file.filename or '').suffix.lower()
    if not suffix:
        raise HTTPException(status_code=400, detail='The uploaded file needs a filename extension such as .tif or .npy.')
    if suffix not in SUPPORTED_IMAGE_EXTENSIONS:
        raise HTTPException(
            status_code=400,
            detail=f'Unsupported format {suffix}. Supported formats: {sorted(SUPPORTED_IMAGE_EXTENSIONS)}.',
        )
    if threshold <= 0 or threshold >= 1:
        raise HTTPException(status_code=400, detail='threshold must be between 0 and 1, for example 0.5.')
    if not MODEL_PATH.exists():
        raise HTTPException(status_code=400, detail=f'Model not found at {MODEL_PATH}. Train or upload a model first.')

    with tempfile.TemporaryDirectory(prefix='cilia_predict_') as tmp:
        image_path = Path(tmp) / f'uploaded{suffix}'
        with image_path.open('wb') as handle:
            shutil.copyfileobj(file.file, handle)
        try:
            model, checkpoint = load_model(MODEL_PATH)
            image_size = tuple(checkpoint.get('image_size', IMAGE_SIZE))
            channel2 = preprocess_image_file(image_path, image_size=image_size, channel_index=CHANNEL_INDEX)
            _, pred_mask = predict_preprocessed(model, channel2, threshold=threshold)
        except Exception as exc:
            raise HTTPException(
                status_code=400,
                detail=(
                    'Prediction failed. Please upload a supported multichannel image in HWC or CHW order with at least '
                    f'two channels, because CHANNEL_INDEX={CHANNEL_INDEX} selects Channel 2. Details: {exc}'
                ),
            )

        mask_png = mask_png_bytes(pred_mask)
        if not return_overlay:
            return StreamingResponse(
                io.BytesIO(mask_png),
                media_type='image/png',
                headers={'Content-Disposition': 'attachment; filename="predicted_cilia_mask.png"'},
            )

        overlay_png = overlay_png_bytes(channel2, pred_mask)
        archive = io.BytesIO()
        with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
            zf.writestr('predicted_cilia_mask.png', mask_png)
            zf.writestr('channel2_predicted_overlay.png', overlay_png)
        archive.seek(0)
        return StreamingResponse(
            archive,
            media_type='application/zip',
            headers={'Content-Disposition': 'attachment; filename="cilia_prediction.zip"'},
        )


@app.post('/train')
async def api_train(
    zip_file: UploadFile = File(...),
    epochs: int = Form(...),
    batch_size: int = Form(...),
    learning_rate: float = Form(1e-4),
    val_fraction: float = Form(0.2),
    confirm_train: bool = Form(False),
    save_model_path: str | None = Form(None),
    overwrite_model: bool = Form(False),
):
    """Fine-tune the existing one-channel U-Net from a ZIP of paired images and masks.

    The ZIP must contain folders named images/ and masks/. Files are matched by filename stem, for example
    images/sample_001.tif and masks/sample_001.tif. Training images may be multichannel HWC or CHW, but only
    zero-based channel index 1 is extracted. Masks should be 2D binary cilia masks.
    """
    if Path(zip_file.filename or '').suffix.lower() != '.zip':
        raise HTTPException(status_code=400, detail='Please upload a .zip file containing images/ and masks/ folders.')
    if not confirm_train:
        raise HTTPException(
            status_code=400,
            detail='Training not started. Resubmit with confirm_train=true plus explicit epochs and batch_size values.',
        )
    if epochs <= 0:
        raise HTTPException(status_code=400, detail='epochs must be a positive integer.')
    if batch_size <= 0:
        raise HTTPException(status_code=400, detail='batch_size must be a positive integer.')
    if learning_rate <= 0:
        raise HTTPException(status_code=400, detail='learning_rate must be positive, for example 0.0001.')
    if not (0 <= val_fraction < 1):
        raise HTTPException(status_code=400, detail='val_fraction must be at least 0 and less than 1, for example 0.2.')
    if not MODEL_PATH.exists():
        raise HTTPException(status_code=400, detail=f'Existing model not found at {MODEL_PATH}. Train an initial model first.')

    target_model = Path(save_model_path) if save_model_path else PROJECT_DIR / f'checkpoints/channel2_cilia_unet_finetuned_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pt'
    if target_model.exists() and not overwrite_model:
        raise HTTPException(
            status_code=400,
            detail=f'Refusing to overwrite {target_model}. Set overwrite_model=true or choose a new save_model_path.',
        )

    with tempfile.TemporaryDirectory(prefix='cilia_train_') as tmp:
        tmp = Path(tmp)
        uploaded_zip = tmp / 'training_data.zip'
        with uploaded_zip.open('wb') as handle:
            shutil.copyfileobj(zip_file.file, handle)
        try:
            extract_dir = tmp / 'training_data'
            extract_dir.mkdir()
            safe_extract_zip(uploaded_zip, extract_dir)
            image_dir, mask_dir = find_training_dirs(extract_dir)
            X_train, y_train, train_pairs = load_paired_training_images_and_masks(
                image_dir=image_dir,
                mask_dir=mask_dir,
                image_size=IMAGE_SIZE,
                channel_index=CHANNEL_INDEX,
            )
            existing_model, _ = load_model(MODEL_PATH)
            _, fine_tune_history = train_model(
                X_train,
                y_train,
                epochs=epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                val_fraction=val_fraction,
                model_path=target_model,
                model=existing_model,
                overwrite_model=overwrite_model,
            )
        except HTTPException:
            raise
        except Exception as exc:
            raise HTTPException(
                status_code=400,
                detail=(
                    'Training failed. Expected a ZIP with images/ and masks/ folders, matching filename stems, '
                    'multichannel images, and 2D binary cilia masks. Details: ' + str(exc)
                ),
            )

    return JSONResponse({
        'status': 'ok',
        'message': 'Fine-tuning completed using only zero-based channel index 1.',
        'trained_pairs': len(train_pairs),
        'saved_model_path': str(target_model),
        'final_metrics': fine_tune_history[-1] if fine_tune_history else {},
        'history': fine_tune_history,
    })


## Start the API with ngrok

Run this cell after you have a saved model at `MODEL_PATH`. If ngrok asks for an auth token, create a free ngrok account and run `ngrok.set_auth_token("YOUR_TOKEN")` before opening the tunnel.

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print('Public API URL:', public_url)
print('Health endpoint:', str(public_url) + '/health')

uvicorn.run(app, host='0.0.0.0', port=8000)

## API Examples

Replace `PUBLIC_URL` with the ngrok URL printed above.

### Health

```bash
curl PUBLIC_URL/health
```

### Predict with curl

This returns a ZIP containing `predicted_cilia_mask.png` and `channel2_predicted_overlay.png`.

```bash
curl -X POST "PUBLIC_URL/predict" \
  -F "file=@/path/to/new_multichannel_image.tif" \
  -F "threshold=0.5" \
  -F "return_overlay=true" \
  --output cilia_prediction.zip
```

### Predict with Python requests

```python
import requests

url = "PUBLIC_URL/predict"
with open("/path/to/new_multichannel_image.tif", "rb") as f:
    response = requests.post(
        url,
        files={"file": f},
        data={"threshold": 0.5, "return_overlay": "true"},
    )
response.raise_for_status()
open("cilia_prediction.zip", "wb").write(response.content)
```

### Fine-tune with curl

The ZIP must contain paired folders named `images/` and `masks/`. The masks must be manually prepared cilia masks with matching filename stems.

```bash
curl -X POST "PUBLIC_URL/train" \
  -F "zip_file=@/path/to/training_pairs.zip" \
  -F "epochs=10" \
  -F "batch_size=4" \
  -F "learning_rate=0.0001" \
  -F "confirm_train=true" \
  --output training_metrics.json
```

### Fine-tune with Python requests

```python
import requests

url = "PUBLIC_URL/train"
with open("/path/to/training_pairs.zip", "rb") as f:
    response = requests.post(
        url,
        files={"zip_file": f},
        data={
            "epochs": 10,
            "batch_size": 4,
            "learning_rate": 0.0001,
            "confirm_train": "true",
        },
    )
response.raise_for_status()
print(response.json())
```

Retraining or fine-tuning requires images and matching manually prepared cilia masks. Detection/prediction requires only new multichannel images.